In [ ]:
!pip install category_encoders

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.9/85.9 kB 3.9 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import joblib
import numpy as np

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import Ridge, ElasticNet
from sklearn.ensemble import RandomForestRegressor

import category_encoders as ce

In [ ]:
df = pd.read_csv("/content/bowler_match_final_stage2.csv")
df["date"] = pd.to_datetime(df["date"])

print("STAGE 1 ✅ Dataset loaded")
print("Rows:", df.shape[0])
print("Date range:", df["date"].min(), "→", df["date"].max())


STAGE 1 ✅ Dataset loaded
Rows: 9283
Date range: 2008-04-18 00:00:00 → 2021-04-23 00:00:00


In [ ]:
TARGET = "wicket"
DROP_COLS = [
    "wicket",     # target
    "bowler",     # identifier
    "matchid",    # identifier
    "date"        # only for split
]
X = df.drop(columns=DROP_COLS)
y = df[TARGET]

print("STAGE 2 ✅ Target separated")
print("Feature columns:", X.columns.tolist())


STAGE 2 ✅ Target separated
Feature columns: ['season', 'venue', 'city', 'bowling_team', 'batting_team', 'runs_conceded', 'balls_bowled', 'wides', 'no_balls', 'overs', 'economy', 'avg_wkts_last_5', 'avg_wkts_last_10', 'avg_wkts_at_venue', 'matches_at_venue', 'matches_played', 'career_avg_wickets', 'career_avg_runs_conceded', 'career_avg_overs', 'discipline_score', 'recent_vs_career_form']


In [ ]:
df["season"] = df["season"].astype(str).str.split('/').str[0].astype(int)
X_train = X[df["season"] <= 2020]
X_test  = X[df["season"] >= 2021]

y_train = y[df["season"] <= 2020]
y_test  = y[df["season"] >= 2021]

print("STAGE 3 ✅ Time-based split")
print("Train size:", X_train.shape)
print("Test size :", X_test.shape)

STAGE 3 ✅ Time-based split
Train size: (9081, 21)
Test size : (202, 21)


In [ ]:
categorical_cols = [
    "season",          # treat as categorical
    "venue",
    "city",
    "bowling_team",
    "batting_team"
]
numerical_cols = [
    "avg_wkts_last_5",
    "avg_wkts_last_10",
    "avg_wkts_at_venue",
    "matches_at_venue",
    "matches_played",
    "career_avg_wickets",
    "career_avg_runs_conceded",
    "career_avg_overs",
    "discipline_score",
    "recent_vs_career_form"
]


In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", ce.TargetEncoder(), categorical_cols),
        ("num", StandardScaler(), numerical_cols)
    ]
)

feature_pipeline = Pipeline(
    steps=[("preprocessing", preprocessor)]
)

feature_pipeline.fit(X_train, y_train)

joblib.dump(feature_pipeline, "bowler_feature_pipeline.pkl")

print("STAGE 5 ✅ Bowler feature pipeline saved")

STAGE 5 ✅ Bowler feature pipeline saved


In [ ]:
X_train_t = feature_pipeline.transform(X_train)
X_test_t  = feature_pipeline.transform(X_test)

print("STAGE 6 ✅ Data transformed")
print("Train shape:", X_train_t.shape)
print("Test shape :", X_test_t.shape)

STAGE 6 ✅ Data transformed
Train shape: (9081, 15)
Test shape : (202, 15)


In [ ]:
baseline_pred = X_test["avg_wkts_last_10"]
baseline_mae = mean_absolute_error(y_test, baseline_pred)

print("STAGE 7 ✅ Baseline evaluated")
print("Baseline MAE:", round(baseline_mae, 3))

STAGE 7 ✅ Baseline evaluated
Baseline MAE: 0.945


In [ ]:
RANDOM_STATE = 42
models = {
    "Ridge": Ridge(alpha=1.0),
    "RandomForest": RandomForestRegressor(
        n_estimators=400,
        max_depth=10,
        min_samples_leaf=10,
        random_state=RANDOM_STATE,
        n_jobs=-1
    )
}

In [ ]:
results = []

# Baseline
results.append(("Baseline", baseline_mae))

for name, model in models.items():
    model.fit(X_train_t, y_train)
    preds = model.predict(X_test_t)
    mae = mean_absolute_error(y_test, preds)
    results.append((name, mae))

In [ ]:
results_df = pd.DataFrame(results, columns=["Model", "MAE"]).sort_values("MAE")

print("\nSTAGE 9 🔍 Model comparison")
print(results_df)


STAGE 9 🔍 Model comparison
          Model       MAE
2  RandomForest  0.875692
1         Ridge  0.876370
0      Baseline  0.944888


In [ ]:
best_model_name, best_mae = results_df.iloc[0]

print("\nBest model:", best_model_name)
print("Best MAE:", round(best_mae, 3))
print("Baseline MAE:", round(baseline_mae, 3))

if best_mae < baseline_mae - 0.2:
    print("✅ Meaningful improvement → proceed with this model")
else:
    print("⚠️ Marginal improvement → baseline still strong")


Best model: RandomForest
Best MAE: 0.876
Baseline MAE: 0.945
⚠️ Marginal improvement → baseline still strong


In [ ]:
final_model = models[best_model_name]
joblib.dump(final_model, "bowler_final_model.pkl")

print(" ✅ Final model saved:", best_model_name)


 ✅ Final model saved: RandomForest
